# Exploratory cloud-top phase: HARP2 GPC V4.0 + OCI L1C V3.0

Pair the HARP2 liquid index with OCI SWIR reflectances, build a joint histogram, and interactively change the spectral-ratio threshold while a three-color global map updates. Default dates are **September 3–5, 2024 inclusive UTC**. Change dates below for another interval, including a full month.

**Your two clarifications are incorporated:** OCI L1C V3.0 is paired with HARP2 GPC V4.0, and the all-liquid simulated reference is a configurable input. No reference simulation has been invented. Until you supply one, the notebook runs in visibly labeled **raw-ratio exploration mode**. Its classifications are provisional candidates, not a reproduction of the ACTIVATE normalized-ratio retrieval.

Install the repository package in the notebook kernel with `python -m pip install -e ".[validation]"` from the repository root. Configure paths as described in the root README. The original liquid-index script is not modified or required. Satellite extraction runs in AWS **us-west-2**; once cached, the histogram, reference changes and map updates can run offline.

In [ ]:
# Install into this notebook kernel; restart the kernel if required.
# Install the package from its repository root; see README.

In [ ]:
from pathlib import Path
import json
import numpy as np
import xarray as xr
import earthaccess
from pace_specpol import matching as mp
from pace_specpol.reference import Reference
from pace_specpol.validation.aggregation import IndexConfig, PhaseIndex, build_index
from pace_specpol.validation.vis_dashboard import dashboard, reference_dashboard
from pace_specpol.validation.vis_samples import plot_cached_samples
from pace_specpol.paths import load_paths
PATHS = load_paths()

cfg = mp.MatchConfig(
    start_date="2024-09-03",
    end_date="2024-09-05",  # includes all of this UTC date
    cache_dir=str(PATHS["pair_cache"]),
    workers=2,
    batch_size=4,
    max_solar_zenith=75.0,
    max_sensor_zenith=65.0,
    min_rho1615=0.01,
    cloud_reject_bits=1,
    min_cloud_optical_thickness=None,
    oci_qc_policy="ignore",  # see the QC discussion below
)
cfg.validate()



## Observable and phase regions

The plotted spectral index is

\[
R = \frac{\rho_{2260}/\rho_{1615}}{(\rho_{2260}/\rho_{1615})_{\mathrm{all\,liquid,sim}}}.
\]

OCI uses a nominal **1615 nm** band, not RSP's 1590 nm band. The inspected OCI file has actual centers **1618.034 and 2258.429 nm**. Both reflectances are selected from the same OCI tilt view. L1C stores radiance, so the code uses its band-specific solar irradiance, solar zenith and Earth–Sun distance to form TOA reflectance. Solar/geometric common factors cancel in the same-view reflectance ratio, but the band-specific solar irradiances do not.

The phase regions follow your figure exactly, with initial ratio cutoff **1.27** and LI cutoff **0.3**:

| Matched observation | Candidate phase |
|---|---|
| Ratio < cutoff, at any LI | Liquid |
| Ratio ≥ cutoff and LI < 0.3 | Ice |
| Ratio ≥ cutoff and LI ≥ 0.3 | Liquid-topped mixed phase (LTMP) |

The low-LI/low-ratio quadrant is therefore assigned **liquid by the spectral rule**, not confirmed liquid by polarimetry. This choice matches the supplied diagram. The raw-ratio mode does **not** make 1.27 equivalent to the normalized ACTIVATE threshold.

Classify each matched observation first, then count classes in geographic cells. The map displays the **most frequent class**, which need not exceed 50%. Separate ice and liquid samples are never merged into an LTMP sample merely by averaging their observables. Cells with no samples, too few samples, or a tie are white. Saved phase counts and fractions preserve the population mixture.

## Pair discovery and cloud extraction (run once)

CMR was checked on September 21, 2026: **437 HARP2 V4.0 granules and 437 OCI L1C V3 granules** were found for the default interval, with matching filename timestamps. The code uses clean S3 links and independently verifies actual grid coordinates and row times, so matching array sizes alone are not sufficient. There is no nearest-day or nearest-granule fallback.

OCI L1C and HARP2 GPC inherit a common swath grid. The tested pair has exactly equal latitudes and longitudes. This is **surface-grid alignment**, not a cloud-top parallax correction. Cloud evolution, different optical sampling depths, and viewing geometry remain scientific limitations.

If both OCI views are valid, the view with the smaller absolute offset from the common nadir time is used once. The code never takes the numerator and denominator from different views. Both HARP2 nadir time and OCI observation time must fall within the requested UTC interval.

The cached files retain LI, both reflectances, raw ratio, viewing geometry, optical thickness, cloud-bow effective radius, coordinates, times and QC availability. You can change the reference or histogram configuration later without returning to NASA data.

**I/O note:** the tested OCI L1C file is approximately 951 MB with contiguous spectral storage. Selecting two channels reduces retained data and computation but may still read a substantial fraction of the source object. Extraction is bounded to a small number of files at a time. Avoid assuming two bands cost only 2/286 of the I/O. The code refreshes Earthdata S3 sessions between batches after 40 minutes; reduce batch size if a batch approaches 20 minutes.

### Explicit initial screening

- Finite LI within −10 to 10, valid paired coordinates and row times.
- A finite HARP2 `cloud_quality` value is required. By the field description, fill indicates no cloud or no cloud determination. Bit 0 (disagreement between cloud-presence tests) is rejected by default. **No cloud-bow/RFT success mask or LI-based liquid-only selection is applied.**
- Both OCI radiances must be positive and finite, observation count positive, view-time offset ≤200 s, solar zenith ≤75°, and sensor zenith ≤65°.
- Denominator reflectance must be ≥0.01; both reflectances must be ≤3. These are configurable exploratory screens, not validated mixed-phase QA.
- No optical-thickness minimum is applied by default. Set `min_cloud_optical_thickness=5` for a sensitivity run if scientifically appropriate; that selection can alter the phase population.

**OCI QC limitation:** in the inspected V3 L1C granule, both selected channels' `qc` values are entirely fill, including where radiances exist. Default `oci_qc_policy="ignore"` means *unverified spectral QC*, not good QC. Availability is saved in every matched sample and summarized below. `"require_zero"` enforces finite zero QC but removes all samples in the inspected file. There is no silent policy fallback. A future validated retrieval should establish usable spectral QA, for example from corresponding OCI L1B quality information.

In [ ]:
# Existing completed caches are reused without an Earthdata request.
# To change extraction settings, use a NEW cache_dir.
manifest_path = Path(cfg.cache_dir) / "manifest.json"
existing = json.loads(manifest_path.read_text()) if manifest_path.exists() else None
if existing and existing.get("complete"):
    if existing["fingerprint"] != cfg.fingerprint:
        raise ValueError("Extraction settings changed: choose a new cache_dir")
    paths = [str(Path(cfg.cache_dir) / name) for name in existing["files"]]
    print(f"Reusing {len(paths)} cached matched granules")
else:
    earthaccess.login()
    pairs, catalogs, discovery_report = mp.discover_pairs(cfg)
    paths = mp.cache_pairs(pairs, catalogs, cfg, discovery_report)

manifest = json.loads(manifest_path.read_text())
records = manifest["records"]
print("Matched samples:", sum(r["matched_samples"] for r in records))
print("Matched samples with unknown OCI QC:",
      sum(r["matched_samples_unknown_oci_qc"] for r in records))
print("First-pair details:", records[0])


## Configure the all-liquid reference

Three modes are implemented:

1. **Raw:** no normalization. Useful for checking coverage, histogram shape and the UI, but cannot transfer the normalized ACTIVATE 1.27 threshold scientifically.
2. **Scalar:** divide every observed ratio by your positive simulated all-liquid ratio. This assumes one reference is appropriate for all selected geometries/cloud properties; document those assumptions.
3. **LUT:** interpolate your simulated ratio by any nonempty subset of `sza`, `vza`, `raa`, `cot`, `re`. There is **no extrapolation**: out-of-domain or missing inputs are excluded and counted.

A LUT uses an xarray NetCDF variable `liquid_reflectance_ratio`, strictly increasing coordinates with at least two values each, and global attributes `numerator_nominal_nm=2260`, `denominator_nominal_nm=1615`. Angles are degrees; `raa` is folded to 0–180°; `cot` is dimensionless; `re` is micrometers. See `../../Documentation/reference_inputs.md` for a template.

Simulations should be consistent with OCI spectral response functions, the TOA definition of these observations, geometry, atmospheric absorption and the chosen cloud microphysics. A LUT requiring cloud-bow `re` will preferentially exclude observations without a successful liquid-droplet retrieval—including potential ice cases. Use such a LUT only with an explicit strategy for that selection bias; missing values are not guessed.

In [ ]:
# Default: explore the observed ratio without claiming ACTIVATE-equivalent normalization.
reference = Reference(mode="raw")

# Option A — replace YOUR_VALUE with your simulated all-liquid rho2260/rho1615:
# reference = Reference(
#     mode="scalar", scalar=YOUR_VALUE,
#     description="OCI-bandpass TOA simulation; specify model, geometry and cloud assumptions",
# )

# Option B — supply a NetCDF LUT:
# reference = Reference(
#     mode="lut", lut_path="my_oci_all_liquid_ratio.nc",
#     description="Specify simulation model, atmosphere, bandpasses and cloud assumptions",
# )

index_cfg = IndexConfig(
    resolution=0.1,       # change to 0.5 or 1.0 for a smaller index
    li_threshold=0.3,
    ratio_min=0.5,
    ratio_max=3.5,
    ratio_step=0.01,      # exact supported slider cuts, not rounded sample values
    histogram_li_min=0.0,
    histogram_li_max=10.0,
    histogram_li_bins=100,
)
index = build_index(cfg.cache_dir, reference, index_cfg)
print("Reusable index:", index.directory)
print(index.metadata["totals"])


The installed package supplies the processing and visualization modules. Restart the kernel after updating it. Existing completed paired caches remain reusable; see the migration documentation.


In [ ]:
viewer = dashboard(
    index,
    threshold=1.27,
    min_count=1,
    output_dir=str(PATHS["export_root"]),
    preview_width=1000,
    ratio_clim=(0.5, 2.0),
    li_clim=(-0.5, 4.0),
)


## Optional orbit mosaic without geographic averaging

These plots draw every eligible cached sample center without 0.1° binning. They still use NASA's approximately 5.2 km L1C sampling. Marker sizes are visual, not physical footprints; overlapping points overplot in manifest order. These are sample-location mosaics, not unique-time or footprint-exact imagery. Plotting millions of points can be slower than the dashboard. No Earthdata access is needed.


In [ ]:
# Uncomment one plot at a time when needed.
# fig, ax = plot_cached_samples(cfg.cache_dir, field="liquid_index", reference=reference)
# fig, ax = plot_cached_samples(cfg.cache_dir, field="ratio", reference=reference)
# fig, ax = plot_cached_samples(cfg.cache_dir, field="phase", reference=reference, ratio_threshold=1.27)
# fig.savefig("mixed_phase_results/sample_mosaic.png", dpi=250, bbox_inches="tight")

print(index.sampling_summary())


## Reopen or export without extraction

After a kernel restart, load the saved index directory and recreate the viewer. It embeds the reference, grid and screening provenance. To change the reference or LI cutoff, rerun `build_index` from cached matched observations; this still performs no satellite I/O. To change extraction QA, date range or geometry screens, create a separate extraction cache.

In [ ]:
# Examples — no new satellite access:
# index = PhaseIndex.load("mixed_phase_cache_20240903_20240905/index_YOUR_ID")
# viewer = dashboard(index, threshold=1.40)
# index.export("mixed_phase_results/phase_cut_1p40.nc", threshold=1.40, min_count=3)

# In the current session, a programmatic change triggers the same update:
# viewer["slider"].value = 1.40


## Interpretation and validation scope

The label **LTMP** means the *combined observable falls in the supplied LTMP region*. It does not by itself validate vertically coexisting ice and liquid, distinguish every multilayer cloud, or remove effects of parallax, cloud heterogeneity, gas absorption, instrument noise or spectral-response differences. The 1.27 cutoff is supplied by your ACTIVATE example and remains provisional for OCI/HARP2.

The full September 3–5 extraction has not been run here. Development checks used a real V4.0 HARP2 granule and the corresponding real OCI V3 L1C SWIR subset. They verified radiance-to-reflectance ratio conversion, grid alignment, time selection, missing QC behavior, phase thresholds, sparse counts against direct sample classification, normalization, LUT limits, NetCDF export and widget callbacks. See `VALIDATION.md` for results and what remains untested.

The existing LI-only L3 mean cannot reconstruct this joint distribution: you need the paired observations generated by this notebook.

### Sources

- [NASA PACE L1C Users Guide](https://oceancolor.gsfc.nasa.gov/files/PACE_L1C_Users_Guide.pdf): shared grid, geometry, radiometry and OCI tilt views.
- [NASA OCI instrument requirements](https://pace.oceansciences.org/requirements.htm): nominal SWIR channels.
- [NASA HARP2 cloud product notebook](https://nasa.github.io/oceandata-notebooks/notebooks/harp2/harp2_l2_cloud_gpc_product.html): LI and cloud-quality fields.
- [NASA processing-version overview](https://nasa.github.io/oceandata-notebooks/notebooks/release_note_highlights.html): separate OCI and HARP2 releases.
- [HARP2 GPC V4 collection](https://search.earthdata.nasa.gov/search/granules?p=C4126552781-OB_CLOUD).
- [OCI L1C V3 collection](https://search.earthdata.nasa.gov/search/granules?p=C3392966961-OB_CLOUD).

The 1.27 phase partition and reference diagram were provided by you; the workflow does not claim that this threshold has already been validated for PACE.